# FoxTrack 04: Time confound test

In notebook 03, the gradient-boosted trees gave alerts on benign packets long before the attack start in two captures. The probable cause: in each capture the attack comes late, so a model can learn the time in the capture and not the attack.

**The question:** Did the model learn what an attack looks like, or did it only learn that attacks come near the end of each recording?

Set `VARIANT` and run all cells. Each variant is one run.

| `VARIANT` | What it does |
|---|---|
| `clock` | Trains a model on benign packets only, to tell early packets from late packets. Tests it on a capture that it did not see. A high ROC-AUC shows that the features contain a clock. Also compares the important features of the clock model and the attack model. |
| `drop_clock` | Repeats notebook 03 for the trees, without the features that the clock model uses most. |
| `time_matched` | Repeats notebook 03 for the trees, with benign training packets only from the attack period. Benign and malicious packets then come from the same time, so the time gives no information. |

**Before you run it:** add the output of notebook 01 in **Input > Add Input > Notebook Output**, and set **Internet** to on.

MLflow records each run in the file `mlflow.db`, in the experiment `time_confound_test`.

In [1]:
# Settings.
VARIANT = "time_matched"                      # "clock", "drop_clock", or "time_matched"
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/confound"
N_PER_CLASS = 100_000
GBT = {"maxIter": 50, "maxDepth": 5}
CLOCK_SHARE = 0.8                      # drop the features that have this share of the clock importance
TARGET_FPRS = (1e-2, 1e-3, 1e-4)
ALERT_RULES = ((1, 1), (3, 5), (10, 20))
SMOOTH_WINDOW = 100                    # packets in the moving mean of the score
assert VARIANT in ("clock", "drop_clock", "time_matched")

In [2]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import etl, split, supervised
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

Code version: 445264e Record the confound test in MLflow and add a test of other settings


In [3]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

Parquet data: /kaggle/input/notebooks/<kaggle-user>/notebookabc04dfe5a/kitsune_parquet


In [4]:
import os, time
import mlflow
import pandas as pd
from foxtrack.anomaly import feature_columns

mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("time_confound_test")

spark = etl.build_spark(driver_memory="20g")
summary = split.load_summary(PARQUET_ROOT)
os.makedirs(f"{OUT_DIR}/scores", exist_ok=True)
ALL_FEATURES = feature_columns(spark.read.parquet(f"{PARQUET_ROOT}/features/attack={summary[0]['attack']}"))
features = ALL_FEATURES
print("Variant:", VARIANT, "| features:", len(ALL_FEATURES))

2026/11/25 20:19:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/25 20:19:15 INFO mlflow.store.db.utils: Updating database tables
2026/11/25 20:19:20 INFO mlflow.tracking.fluent: Experiment with name 'time_confound_test' does not exist. Creating a new experiment.
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 20:19:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Variant: time_matched | features: 115


## 1. Clock model

Benign packets before the attack only. Label 1 = the last 15% of the benign period.

In [5]:
%%time
if VARIANT in ("clock", "drop_clock"):
    path = f"{OUT_DIR}/clock_sample"
    supervised.build_clock_sample(spark, PARQUET_ROOT, summary, N_PER_CLASS).write.mode("overwrite").parquet(path)
    clock = supervised.assemble(spark.read.parquet(path)).cache()
    clock_model = supervised.make_estimator("gbt", **GBT).fit(clock)
    clock_importance = supervised.feature_importance(clock_model, ALL_FEATURES)
    clock_importance.to_csv(f"{OUT_DIR}/clock_importance.csv")
    print("### CLOCK_IMPORTANCE"); print(clock_importance.head(25).round(4).to_csv())
    with mlflow.start_run(run_name=f"{VARIANT}_clock_model"):
        mlflow.log_params({"variant": VARIANT, "model": "gbt_clock", "n_per_class": N_PER_CLASS, **GBT})
        mlflow.log_artifact(f"{OUT_DIR}/clock_importance.csv", artifact_path="importance")

if VARIANT == "clock":
    rows = []
    for row in summary:
        start = time.time()
        auc = supervised.clock_fold_auc(clock, row["attack"], maxIter=20, maxDepth=5)
        rows.append({"attack": row["attack"], "clock_roc_auc": auc})
        print(f"{row['attack']}: clock ROC-AUC {auc:.3f} ({time.time() - start:.0f} s)")
    clock_auc = pd.DataFrame(rows)
    clock_auc.to_csv(f"{OUT_DIR}/clock_auc.csv", index=False)
    print("### CLOCK_AUC"); print(clock_auc.to_csv(index=False))
    with mlflow.start_run(run_name="clock_leave_one_out"):
        mlflow.log_params({"variant": VARIANT, "model": "gbt_clock", "n_per_class": N_PER_CLASS, "maxIter": 20, "maxDepth": 5})
        for r in clock_auc.itertuples():
            mlflow.log_metric(f"clock_roc_auc_{r.attack}", r.clock_roc_auc)
        mlflow.log_metric("clock_roc_auc_median", clock_auc["clock_roc_auc"].median())
        mlflow.log_artifact(f"{OUT_DIR}/clock_auc.csv", artifact_path="results")

if VARIANT == "drop_clock":
    dropped = supervised.top_features(clock_importance, share=CLOCK_SHARE)
    features = [f for f in ALL_FEATURES if f not in dropped]
    print("### DROPPED", ",".join(dropped))
    with mlflow.start_run(run_name="dropped_clock_features"):
        mlflow.log_params({"variant": VARIANT, "clock_share": CLOCK_SHARE, "dropped_features": ",".join(dropped),
                           "features_remaining": len(features)})
    print(f"Dropped {len(dropped)} clock features. {len(features)} features remain.")

CPU times: user 6 µs, sys: 1e+03 ns, total: 7 µs
Wall time: 14.3 µs


## 2. Attack model

For `clock`: one attack model on all captures, to compare its important features with the clock model.
For the other variants: the leave-one-capture-out test of notebook 03.

In [6]:
%%time
period = "attack" if VARIANT == "time_matched" else "all"
sample_path = f"{OUT_DIR}/train_sample"
supervised.build_training_sample(spark, PARQUET_ROOT, summary, N_PER_CLASS, benign_period=period).write.mode("overwrite").parquet(sample_path)
train = supervised.assemble(spark.read.parquet(sample_path), features).cache()
train.groupBy("attack").pivot("label").count().orderBy("attack").show()

if VARIANT == "clock":
    attack_model = supervised.make_estimator("gbt", **GBT).fit(train)
    attack_importance = supervised.feature_importance(attack_model, ALL_FEATURES)
    attack_importance.to_csv(f"{OUT_DIR}/attack_importance.csv")
    print("### ATTACK_IMPORTANCE"); print(attack_importance.head(25).round(4).to_csv())
    clock_top = supervised.top_features(clock_importance, share=CLOCK_SHARE)
    share = attack_importance[clock_top].sum()
    print(f"### OVERLAP The {len(clock_top)} top clock features have {share:.1%} of the attack model importance.")
    with mlflow.start_run(run_name="attack_model_importance"):
        mlflow.log_params({"variant": VARIANT, "model": "gbt", "n_per_class": N_PER_CLASS, **GBT})
        mlflow.log_metric("clock_features_in_top_set", len(clock_top))
        mlflow.log_metric("attack_importance_share_of_clock_features", float(share))
        mlflow.log_artifact(f"{OUT_DIR}/attack_importance.csv", artifact_path="importance")
else:
    all_results = []
    for row in summary:
        s = split.onset_split(row)
        start = time.time()
        capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
        results, test = supervised.run_fold(train, capture, s, "gbt", TARGET_FPRS, ALERT_RULES, features=features, smooth_windows=(1, SMOOTH_WINDOW), **GBT)
        results["model"] = f"gbt_{VARIANT}"
        all_results.append(results)
        test.to_parquet(f"{OUT_DIR}/scores/gbt_{VARIANT}_{s.attack}.parquet", index=False)
        with mlflow.start_run(run_name=f"gbt_{VARIANT}_{s.attack}"):
            mlflow.log_params({"variant": VARIANT, "test_capture": s.attack, "model": "gbt", "n_per_class": N_PER_CLASS,
                               "features": len(features), "benign_period": period, **GBT})
            mlflow.log_metric("roc_auc", results["roc_auc"].iloc[0])
            mlflow.log_metric("pr_auc", results["pr_auc"].iloc[0])
            for r in results.itertuples():
                tag = f"fpr{r.target_fpr:g}_{r.alert_rule.replace(' ', '_')}"
                mlflow.log_metric(f"false_alarm_events_{tag}", r.false_alarm_events)
                mlflow.log_metric(f"packet_recall_{tag}", r.packet_recall)
                if r.detected:
                    mlflow.log_metric(f"delay_packets_{tag}", r.delay_packets)
        print(f"{s.attack}: done in {time.time() - start:.0f} s")
    results = pd.concat(all_results, ignore_index=True)
    results.to_csv(f"{OUT_DIR}/results.csv", index=False)
    print("### RESULTS"); print(results.to_csv(index=False))

26/11/25 20:19:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+------+------+
|           attack|     0|     1|
+-----------------+------+------+
|   active_wiretap| 55473|100000|
|         arp_mitm| 58995|100000|
|          fuzzing|100000|100000|
|          os_scan|100000| 65700|
|       ssdp_flood| 24220|100000|
|ssl_renegotiation|100000| 92652|
|          syn_dos| 81889|  7038|
|  video_injection|100000|100000|
+-----------------+------+------+

arp_mitm: done in 199 s
active_wiretap: done in 189 s
fuzzing: done in 179 s
mirai_botnet: done in 196 s
os_scan: done in 177 s
ssdp_flood: done in 187 s
ssl_renegotiation: done in 182 s
syn_dos: done in 181 s
video_injection: done in 167 s
### RESULTS
attack,model,target_fpr,alert_rule,smooth_window,threshold,k,n,detected,delay_packets,benign_packets_before_attack,false_alarms_before_attack,false_alarms_per_100k,false_alarm_events,false_alarm_events_per_100k,false_alarm_events_per_100k_upper95,packet_recall,alert_lift,detection_credible,chance_delay_probability,delay_credible,placebo